In [5]:
import os
import kagglehub
from PIL import Image
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from pathlib import Path

In [2]:
os.environ["KAGGLEHUB_CACHE"] = "data/kagglehub_cache"

path = kagglehub.dataset_download(
    "preetviradiya/brian-tumor-dataset"
)
print(path)

data/kagglehub_cache\datasets\preetviradiya\brian-tumor-dataset\versions\1


In [11]:
def load_images_from_folders(root_path):
    images = []
    labels = []

    class_map = {
        "Brain Tumor": 1,
        "Healthy": 0
    }

    for class_name, label in class_map.items():
        class_path = os.path.join(root_path, class_name)
        for file in os.listdir(class_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')):
                img_path = os.path.join(class_path, file)

                try:
                    img = Image.open(img_path)
                    img = img.convert("RGB")
                    img = img.resize((128, 128))
                    img_array = np.array(img)
                    if img_array.dtype == np.uint16:
                        img_array = img_array.astype(np.float32) / 65535.0
                    else:
                        img_array = img_array.astype(np.float32) / 255.0

                    images.append(img_array)
                    labels.append(label)

                except Exception as e:
                    print(f"Ошибка: {file} -> {e}")

    return np.array(images), np.array(labels)

In [13]:
root_path = Path(path) / "Brain Tumor Data Set" / "Brain Tumor Data Set"
X, Y = load_images_from_folders(
    root_path=root_path
)

print(X.shape)
print(Y.shape)

sss = StratifiedShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(sss.split(X, Y))

x_train, x_test = X[train_idx], X[test_idx]
y_train, y_test = Y[train_idx].astype(np.float32), Y[test_idx].astype(np.float32)

(4600, 128, 128, 3)
(4600,)
